In [1]:
using EphemerisSources
using AstroTime, Dates
using GLMakie
using LinearAlgebra

function horizons_to_matrix(h)
    N = length(h.x)
    T = eltype(h.x)

    # 7×N を1回だけ確保（ゼロ初期化しないので高速）
    A = Matrix{T}(undef, 6, N)

    # @views：A[1,:] などの部分参照で余計なコピーを作らない
    @views begin
        A[1, :] .= h.x
        A[2, :] .= h.y
        A[3, :] .= h.z
        A[4, :] .= h.ẋ
        A[5, :] .= h.ẏ
        A[6, :] .= h.ż
    end

    return A
end


"""
各列ごとに cross(a[:,i], b[:,i]) を計算し、正規化したベクトルを返す関数。

想定する入力:
- a, b : 3×N 行列
  各列が 3 次元ベクトル（例: 位置 r(t), 速度 v(t)）

出力:
- 3×N 行列
  各列が正規化された外積ベクトル（例: 軌道面法線方向）
"""
function calculate_normalized_cross_product(a::AbstractMatrix, b::AbstractMatrix)

    # --- 入力チェック ---
    # @assert を使う目的:
    # ・関数の前提条件（3×N 行列）を明示するため
    # ・間違った次元の入力が来た場合に、静かにバグらず即エラーにするため
    @assert size(a,1) == 3 && size(b,1) == 3 "a and b must be 3×N matrices"
    @assert size(a,2) == size(b,2)           "a and b must have the same number of columns"

    # 時系列長（ベクトルの本数）
    N = size(a,2)

    # 出力行列（3×N）
    result = zeros(eltype(a), 3, N)

    # --- メインループ ---
    # @views を使う目的:
    # ・a[:, i], result[:, i] は通常「コピー」を作る
    # ・@views を付けることで「参照（view）」になり、
    #   余計なメモリアロケーションを防ぎ高速・省メモリになる
    @views for i in 1:N
        # 各時刻 i における外積（3 次元ベクトル）
        c = cross(a[:, i], b[:, i])

        # 正規化して格納
        # ※ a と b が平行（またはゼロ）な場合は norm(c)=0 になる点に注意
        result[:, i] .= c / norm(c)
    end

    return result
end


"""
各列ごとに 2 つの 3 次元ベクトル a[:,i], b[:,i] のなす角 [rad] を計算する関数。
角度は atan2(||a×b||, a⋅b) により求める（0〜π）。

想定入力:
- a, b : 3×N 行列（各列が 1 時刻の 3D ベクトル）

戻り値:
- alpha : 長さ N のベクトル（各時刻の角度 [rad]）
"""
function calculate_angle(a::AbstractMatrix, b::AbstractMatrix;
                               eps::Real = 0.0,
                               zero_behavior::Symbol = :nan)

    # --- 入力の前提条件チェック ---
    # ・各列が 3 次元ベクトルであることを保証
    # ・a と b が同じ時系列長を持つことを保証
    # → 間違った次元入力を早期に検出するため
    @assert size(a,1) == 3 && size(b,1) == 3 "a and b must be 3×N matrices"
    @assert size(a,2) == size(b,2)           "a and b must have the same number of columns"

    # ベクトル（時刻）の本数
    N = size(a,2)

    # 出力の数値型を入力に合わせつつ、角度計算の安定性のため Float64 を含める
    T = promote_type(eltype(a), eltype(b), Float64)

    # 各時刻の角度を格納する配列
    alpha = Vector{T}(undef, N)

    # --- 各時刻ごとの角度計算 ---
    # @inbounds:
    # ・ループ内の配列境界チェックを省略し、高速化
    # ・@assert により安全性はすでに担保されている
    @inbounds for i in 1:N

        # a, b の i 時刻目の 3 次元成分を取り出す
        ax, ay, az = a[1,i], a[2,i], a[3,i]
        bx, by, bz = b[1,i], b[2,i], b[3,i]

        # 各ベクトルのノルム（長さ）を計算
        na = sqrt(ax*ax + ay*ay + az*az)
        nb = sqrt(bx*bx + by*by + bz*bz)

        # ゼロベクトル（または極端に小さい）場合は角度が定義できない
        if na * nb <= eps
            # zero_behavior に応じて 0 または NaN を返す
            alpha[i] = (zero_behavior === :zero) ? zero(T) : T(NaN)
        else
            # 内積 a⋅b（角度の cos 成分に相当）
            d  = ax*bx + ay*by + az*bz

            # 外積 a×b（角度の sin 成分に相当）
            cx = ay*bz - az*by
            cy = az*bx - ax*bz
            cz = ax*by - ay*bx

            # 外積の大きさ ||a×b||
            cn = sqrt(cx*cx + cy*cy + cz*cz)

            # atan2(||a×b||, a⋅b) により角度を計算
            # ・acos に比べて数値的に安定
            # ・0〜π の範囲が自然に得られる
            alpha[i] = atan(cn, d)  # Julia の atan(y, x) = atan2(y, x)
        end
    end

    return alpha
end


"""
Rodrigues の回転公式で、各列ベクトル coordinates[:,i] を
軸 n[:,i]（単位ベクトル）まわりに角度 θ_i だけ回転する。

入力:
- coordinates : 3×N （各列が回転したい 3D ベクトル）
- cosθ, sinθ  : 長さ N のベクトル（各時刻の cos(θ), sin(θ)）
- n           : 3×N （各列が回転軸の単位ベクトル）

出力:
- 3×N （各列が回転後ベクトル）
"""
function rotate_coordinates(coordinates::AbstractMatrix,
                            cosθ::AbstractVector,
                            sinθ::AbstractVector,
                            n::AbstractMatrix)
    # 前提チェック（入力ミスを早期発見する）
    @assert size(coordinates,1) == 3 "coordinates must be 3×N"
    @assert size(n,1) == 3           "n must be 3×N"
    @assert size(coordinates,2) == size(n,2) "coordinates and n must have same N"
    @assert length(cosθ) == size(coordinates,2) "cosθ length must match N"
    @assert length(sinθ) == size(coordinates,2) "sinθ length must match N"

    N = size(coordinates, 2)
    T = promote_type(eltype(coordinates), eltype(n), eltype(cosθ), eltype(sinθ), Float64)
    out = Matrix{T}(undef, 3, N)

    @inbounds for i in 1:N
        # 回転対象ベクトル v
        x, y, z = coordinates[1,i], coordinates[2,i], coordinates[3,i]

        # 回転軸（単位）k = n[:,i]
        nx, ny, nz = n[1,i], n[2,i], n[3,i]

        c = cosθ[i]
        s = sinθ[i]
        one_c = one(T) - c

        # Rodrigues 回転行列の各成分（k は単位ベクトルを想定）
        r11 = nx*nx*one_c + c
        r12 = nx*ny*one_c - nz*s
        r13 = nx*nz*one_c + ny*s

        r21 = ny*nx*one_c + nz*s
        r22 = ny*ny*one_c + c
        r23 = ny*nz*one_c - nx*s

        r31 = nz*nx*one_c - ny*s
        r32 = nz*ny*one_c + nx*s
        r33 = nz*nz*one_c + c

        # v_rot = R * v
        out[1,i] = r11*x + r12*y + r13*z
        out[2,i] = r21*x + r22*y + r23*z
        out[3,i] = r31*x + r32*y + r33*z
    end

    return out
end


"""
Earth–Moon fixed / Sun–Earth fixed 等に使える汎用回転関数（地球中心など）。

入力:
- states : 6×N 行列
    states[1:3,:] … 位置ベクトル r
    states[4:6,:] … 速度ベクトル v

キーワード引数:
- ẑ : 3×N 基準 z 軸（省略時は (0,0,1)）

出力:
- x_rot  : 3×N 最終回転後の位置
"""

function make_line_fixed_frame(states::AbstractMatrix, states_sc::AbstractMatrix;
                               ẑ::Union{Nothing,AbstractMatrix}=nothing)

    # --- 入力チェック ---
    @assert size(states,1) == 6 "states must be 6×N (position + velocity)"
    N = size(states,2)

    # 状態分解
    r = states[1:3, :]   # 位置
    v = states[4:6, :]   # 速度

    # 基準 z 軸
    if ẑ === nothing
        ẑ = repeat([0.0, 0.0, 1.0], 1, N)
    else
        @assert size(ẑ,1) == 3 && size(ẑ,2) == N "ẑ must be 3×N"
    end

    # ------------------------------------------------------------------
    # (1) 軌道面を XY 平面にそろえる回転
    # ------------------------------------------------------------------

    # 軌道面法線（単位）：normal = normalize(r×v)
    normal = calculate_normalized_cross_product(r, v)

    # 回転軸：normal × ẑ
    rot_axis = calculate_normalized_cross_product(normal, ẑ)

    # 回転角：angle(normal, ẑ)
    α = calculate_angle(normal, ẑ)

    # 軌道面合わせ回転（中間生成物）
    x_plane = rotate_coordinates(
        states[1:3, :],
        cos.(α),
        sin.(α),
        rot_axis
    )

    # 軌道面合わせ回転（中間生成物）
    x_plane_sc = rotate_coordinates(
        states_sc[1:3, :],
        cos.(α),
        sin.(α),
        rot_axis
    )

    # ------------------------------------------------------------------
    # (2) z 軸回り回転で「参照方向（r）」を +x 軸へ
    # ------------------------------------------------------------------

    dist = sqrt.(r[1,:].^2 .+ r[2,:].^2 .+ r[3,:].^2)

    cosθ = x_plane[1,:] ./ dist
    sinθ = -x_plane[2,:] ./ dist   # +x 方向へ回す符号（あなたの定義を踏襲）

    x_rot_sc = rotate_coordinates(
        x_plane_sc,
        cosθ,
        sinθ,
        ẑ
    )

    # 符号付き角（ログ・解析用、不要なら消してOK）
    θ = atan.(sinθ, cosθ)

    return x_rot_sc
end

using LinearAlgebra

"""
t=0 の軌道面を基準に、軌道面がほぼ XY 平面に揃うよう固定回転を全時刻へ適用する。

目的:
- 各時刻で「傾斜角=0」にしてしまうと軌道面の自然な時間変動が見えなくなるため、
  t=0 の回転角・回転軸を固定して回転する（可視化用途向け）。

入力:
- states : 6×N（states[1:3,:]=r, states[4:6,:]=v）
キーワード:
- ẑ : 3×N の基準 z 軸（省略時は (0,0,1) を N 回）
出力:
- r_rot : 3×N（固定回転後の位置）
"""
function rotate_to_plane_fixed_t0(states::AbstractMatrix,states_sc::AbstractMatrix;
                                  ẑ::Union{Nothing,AbstractMatrix}=nothing)
    @assert size(states,1) == 6 "states must be 6×N"
    N = size(states,2)

    r = states[1:3, :]
    v = states[4:6, :]

    # 基準 z 軸
    if ẑ === nothing
        ẑ = repeat([0.0, 0.0, 1.0], 1, N)
    else
        @assert size(ẑ,1) == 3 && size(ẑ,2) == N "ẑ must be 3×N"
    end

    # 軌道面法線（単位）
    normal = calculate_normalized_cross_product(r, v)

    # 回転軸（単位）
    rotation_axis = calculate_normalized_cross_product(normal, ẑ)

    # 回転角（各時刻）
    α = calculate_angle(normal, ẑ)

    # t=0 の回転角・回転軸を固定して全時刻へ展開
    α_fixed = fill(α[1], N)                   # 長さ N
    axis_fixed = repeat(rotation_axis[:, 1], 1, N)        # 3×N

    # 固定回転を適用（あなたの正式版 rotate_coordinates に合わせて cos/sin を渡す）
    r_rot_sc = rotate_coordinates(states_sc[1:3, :], cos.(α_fixed), sin.(α_fixed), axis_fixed)

    return r_rot_sc
end

# J2000 黄道傾斜角
const EPS_J2000 = deg2rad(23.439291111)

# 係数は定数化（関数呼び出しごとにcos/sinしない）
const COS_EPS_J2000 = cos(EPS_J2000)
const SIN_EPS_J2000 = sin(EPS_J2000)

function eclJ2000_to_EME2000(states::AbstractMatrix{Float64})
    @assert size(states, 1) == 6 "states must be 6×N (x,y,z,ẋ,ẏ,ż)"
    N = size(states, 2)

    out = Matrix{Float64}(undef, 6, N)

    c = COS_EPS_J2000
    s = SIN_EPS_J2000

    @inbounds for i in 1:N
        # position
        x = states[1,i]; y = states[2,i]; z = states[3,i]
        out[1,i] = x
        out[2,i] =  c*y - s*z
        out[3,i] =  s*y + c*z

        # velocity
        xd = states[4,i]; yd = states[5,i]; zd = states[6,i]
        out[4,i] = xd
        out[5,i] =  c*yd - s*zd
        out[6,i] =  s*yd + c*zd
    end

    return out
end



eclJ2000_to_EME2000 (generic function with 1 method)

https://juliaastro.org/EphemerisSources.jl/docs/stable/examples/telescope-study/
https://juliaastro.org/AstroTime/stable/tutorial/#Tutorial
https://ssd.jpl.nasa.gov/horizons/app.html#/

In [2]:
####################
frame = "J2000"
wrt   = "sun"
units = "KM-S"

####################
#spacecraft = "Orion EM-1"
#start = TDBEpoch(2022, 11, 16, 9, 4, 0.0)
#stop  = TDBEpoch(2022, 12, 11, 17, 21, 0.0)
#step  = "30 min"

####################
#spacecraft = "CAPSTONE"
#start = TDBEpoch(2022, 6, 28, 10, 6, 7.0)
#stop  = start + 150days
#step  = "1 hour"

####################
spacecraft = "CAPSTONE"
start = TDBEpoch(2026, 1, 25, 3, 54, 1.0)
stop  = TDBEpoch(2026, 3,  3, 11, 36, 27.0)
#stop  = start + 150days
step  = "1 hour"
step_size_sec = 60*60

####################
#spacecraft = "Hayabusa 2"
#start = TDBEpoch(2014, 12, 3, 6, 14, 54.0)
#stop  = TDBEpoch(2026, 4, 10, 0, 0, 0.0)
#step  = "1 day"
#ryugu_horizons = ephemeris("162173", start, stop, step ; wrt = wrt, units = units, frame = freme)
#ryugu_states = horizons_to_matrix(ryugu_horizons)

####################
#spacecraft = "Europa Clipper" #全体
#start = TDBEpoch(2024, 10, 14, 16, 15, 4.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#step  = "1 day"
#step  = "2 hour"
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)

####################
#spacecraft = "Europa Clipper" # 木星に入ってから
#start = TDBEpoch(2030, 4, 1, 0, 0, 0.0)
#start = TDBEpoch(2031, 1, 1, 0, 0, 0.0)
#stop  = TDBEpoch(2034, 9, 3, 4, 46, 13.0)
#stop  = TDBEpoch(2032, 1, 1, 0, 0, 0.0)
#step  = "1 hour"
#europa_horizons = ephemeris("502", start, stop, step ; wrt = wrt, units = units, frame = frame)
#europa_states = horizons_to_matrix(europa_horizons)

# TDBで指定する。
earth_horizons = ephemeris("earth", start, stop, step ; wrt = wrt, units = units, frame = frame)
mars_horizons = ephemeris("mars", start, stop, step ; wrt = wrt, units = units, frame = frame)
jupiter_horizons = ephemeris("jupiter", start, stop, step ; wrt = wrt, units = units, frame = frame)
moon_horizons = ephemeris("moon", start, stop, step ; wrt = wrt, units = units, frame = frame)

sc1_horizons  = ephemeris(spacecraft, start, stop, step ; wrt = wrt, units = units, frame = frame)
earth_states = horizons_to_matrix(earth_horizons)
mars_states = horizons_to_matrix(mars_horizons)
jupiter_states = horizons_to_matrix(jupiter_horizons)
moon_states = horizons_to_matrix(moon_horizons)
sc1_states  = horizons_to_matrix(sc1_horizons)

t_tdb       = TDBEpoch.(moon_horizons.t .* days; origin = :julian)
t_tai       = TAIEpoch.(t_tdb)

println(size(moon_states))
println(size(sc1_states))
#println(size(sc2_states))
println(moon_states[1:6,1])
println(sc1_states[1:6,1])
#println(sc2_states[1:6,1])
println(t_tdb[1])
println(t_tai[1])

(6, 896)
(6, 896)
[-8.374343346356623e7, 1.210475054519947e8, 15908.68507739156, -25.35192951456346, -16.17528541721015, 0.06800288388233522]
[-8.374491953403778e7, 1.210648222289621e8, -6429.573876664042, -25.32831475162891, -16.05305523551247, -0.4066347013308942]
2026-01-25T03:54:01.000 TDB
2026-01-25T03:53:28.815 TAI


In [3]:
using DifferentialEquations
using LinearAlgebra

# --- GM [km^3/s^2] ---
const GM_sun   = 1.3271244004193938e11
const GM_earth = 398600.435436
const GM_moon  = 4902.800066
const GM_jupiter  = 126686531.900 #万有引力定数×木星の質量, km^3/s^(-2)
const GM_mars  = 37931206.234#42828.375214


# --- 半径 [km] ---
const earth_r = 6378.137
const moon_r  = 1737.4
const mars_r  = 3396.2
const jupiter_r  = 69911

"""
Ephemeris は 6×N（列が時刻）を想定:
  ephem[1:3, k] が位置, ephem[4:6, k] が速度（速度は今回未使用）
"""
Base.@kwdef struct EphemParams{T, A<:AbstractMatrix{T}, S<:Real}
    earth::A           # 6×N
    moon::A            # 6×N
    mars::A            # 6×N
    jupiter::A         # 6×N
    step_size_sec::S   # IntでもFloatでもOK
    sol_from::Int
end


# --- 線形補間で位置（1:3）を返す（6×N対応）---
@inline function interp_pos_6xN(
    ephem::AbstractMatrix,   # [6×N] エフェメリス行列（1–3行目が位置、列が時系列）
    t::Real,                 # 補間したい時刻 [s]
    step_size_sec::Real,     # エフェメリスのサンプリング間隔 [s]
    sol_from::Int            # エフェメリス開始時刻に対応する列番号（オフセット）
)
    # 時刻 t に対応する「連続的な列インデックス」（整数でなくてよい）
    col = t / step_size_sec + sol_from

    # エフェメリスの総列数 N
    n = size(ephem, 2)

    # 補間に使う左側の列番号（範囲外アクセスを防ぐため clamp）
    c0 = clamp(floor(Int, col), 1, n - 1)

    # 補間に使う右側の列番号
    c1 = c0 + 1

    # 線形補間係数 α（c0 → c1 の間で 0〜1）
    α = col - c0

    # c0 列の位置ベクトル (x, y, z) をビューとして取得（コピーを作らない）
    @views p0 = ephem[1:3, c0]

    # c1 列の位置ベクトル (x, y, z) をビューとして取得
    @views p1 = ephem[1:3, c1]

    # 線形補間: p(t) = p0 + α (p1 - p0)
    return p0 .+ (p1 .- p0) .* α
end


# --- 運動方程式（日心・Ecliptic J2000 前提）---
function spacecraft_motion!(du, u, p::EphemParams, t)
    earth_pos = interp_pos_6xN(p.earth, t, p.step_size_sec, p.sol_from)
    moon_pos  = interp_pos_6xN(p.moon,  t, p.step_size_sec, p.sol_from)
    mars_pos  = interp_pos_6xN(p.mars,  t, p.step_size_sec, p.sol_from)
    jupiter_pos  = interp_pos_6xN(p.jupiter,  t, p.step_size_sec, p.sol_from)

    @views r_sun = u[1:3]              # 太陽原点
    r_earth = r_sun .- earth_pos
    r_moon  = r_sun .- moon_pos
    r_mars  = r_sun .- mars_pos
    r_jupiter  = r_sun .- jupiter_pos

    du[1] = u[4]
    du[2] = u[5]
    du[3] = u[6]

    inv_sun3   = inv(norm(r_sun)^3)
    inv_earth3 = inv(norm(r_earth)^3)
    inv_moon3  = inv(norm(r_moon)^3)
    inv_mars3  = inv(norm(r_mars)^3)
    inv_jupiter3  = inv(norm(r_jupiter)^3)

    du[4] = -(GM_sun*inv_sun3*r_sun[1] + GM_earth*inv_earth3*r_earth[1] + GM_moon*inv_moon3*r_moon[1] + GM_mars*inv_mars3*r_mars[1] + GM_jupiter*inv_jupiter3*r_jupiter[1])
    du[5] = -(GM_sun*inv_sun3*r_sun[2] + GM_earth*inv_earth3*r_earth[2] + GM_moon*inv_moon3*r_moon[2] + GM_mars*inv_mars3*r_mars[2] + GM_jupiter*inv_jupiter3*r_jupiter[2])
    du[6] = -(GM_sun*inv_sun3*r_sun[3] + GM_earth*inv_earth3*r_earth[3] + GM_moon*inv_moon3*r_moon[3] + GM_mars*inv_mars3*r_mars[3] + GM_jupiter*inv_jupiter3*r_jupiter[3])
    return nothing
end

# --- 衝突イベント（ContinuousCallback用に連続値）---
function collision_g(u, t, integrator)
    p = integrator.p
    earth_pos = interp_pos_6xN(p.earth, t, p.step_size_sec, p.sol_from)
    moon_pos  = interp_pos_6xN(p.moon,  t, p.step_size_sec, p.sol_from)
    mars_pos  = interp_pos_6xN(p.mars,  t, p.step_size_sec, p.sol_from)
    jupiter_pos  = interp_pos_6xN(p.jupiter,  t, p.step_size_sec, p.sol_from)

    r = @view u[1:3]
    dE = norm(r .- earth_pos) - earth_r
    dM = norm(r .- moon_pos)  - moon_r
    dMa = norm(r .- mars_pos)  - mars_r
    dJu = norm(r .- jupiter_pos)  - jupiter_r
    return min(dE, dM, dMa, dJu)  # 0 を下回ったら地球or月に侵入
end

# コールバック用の停止関数：
# 衝突条件が満たされた瞬間に積分を強制終了する
stop!(integrator) = terminate!(integrator)

# 連続コールバックの定義：
# collision_g(u, t, integrator) = 0 を検出した時点で stop! を実行
# rootfind = true により、イベント時刻を補間で高精度に特定する
cb = ContinuousCallback(
    collision_g,    # 衝突（イベント）判定関数 g(u, t)
    stop!,          # g = 0 到達時に呼ばれる処理（積分停止）
    rootfind = true # ルート探索により正確なイベント時刻を検出
)


function solve_satellite(sc1::AbstractMatrix, earth::AbstractMatrix, moon::AbstractMatrix, mars::AbstractMatrix,jupiter::AbstractMatrix,;
                         sol_from::Int,
                         step_size_sec::Real,
                         reltol=1e-10, abstol=1e-10)

    @views u0 = vec(sc1[1:6, sol_from])

    # エフェメリスの有効範囲から tspan を自動決定（c1=c0+1を使うため -1）
    Nmin = min(size(earth,2), size(moon,2), size(sc1,2))
    tmax = (Nmin - sol_from - 1) * step_size_sec
    tspan = (0.0, tmax)

    p = EphemParams(earth=earth, moon=moon, mars=mars, jupiter=jupiter, step_size_sec=step_size_sec, sol_from=sol_from)
    prob = ODEProblem(spacecraft_motion!, u0, tspan, p)

    ts  = 0.0:step_size_sec:tmax
    sol = solve(prob, Vern7();
                callback=cb,
                reltol=reltol, abstol=abstol,
                saveat=ts)

    return sol
end



solve_satellite (generic function with 1 method)

In [4]:
# EME2000への座標変換
moon_states_eme = eclJ2000_to_EME2000(moon_states-earth_states)
sc1_states_eme  = eclJ2000_to_EME2000(sc1_states-earth_states)
#sc2_states_eme  = eclJ2000_to_EME2000(sc2_states-earth_states)

# 地球月固定回転座標系(地球中心)への変換
moon_em_fixed_e = make_line_fixed_frame(moon_states-earth_states, moon_states-earth_states)
sc1_em_fixed_e  = make_line_fixed_frame(moon_states-earth_states, sc1_states-earth_states)
#sc2_em_fixed_e  = make_line_fixed_frame(moon_states-earth_states, sc2_states-earth_states)

# 地球月固定回転座標系(月中心)への変換
sc1_em_fixed_m  = sc1_em_fixed_e - moon_em_fixed_e
#sc2_em_fixed_m  = sc2_em_fixed_e - moon_em_fixed_e

# 太陽地球固定回転座標系への変換
earth_se_fixed_s = make_line_fixed_frame(earth_states, earth_states)
moon_se_fixed_e = make_line_fixed_frame(earth_states, moon_states) - earth_se_fixed_s
sc1_se_fixed_e  = make_line_fixed_frame(earth_states, sc1_states) - earth_se_fixed_s
#sc2_se_fixed_e  = make_line_fixed_frame(earth_states, sc2_states) - earth_se_fixed_s

# 白道面への変換
moon_lop_e = rotate_to_plane_fixed_t0(moon_states-earth_states, moon_states-earth_states) #lunar orbital plane
sc1_lop_e  = rotate_to_plane_fixed_t0(moon_states-earth_states, sc1_states-earth_states)
#sc2_lop_e  = rotate_to_plane_fixed_t0(moon_states-earth_states, sc2_states-earth_states)


3×896 Matrix{Float64}:
      3.46413e5  344932.0        …      -3.66598e5      -3.67805e5
      1.53041e5       1.56758e5          1.36754e5       1.33378e5
 -23663.8        -25372.2           -51118.2        -50216.9

In [5]:
function traj_fade!(
    ax,
    pos_xyz,
    frame_0::Observable{Int};
    trail_len::Int = 800,
    linewidth = 1,
    color = :lime,
    markersize = 10
)
    # 位置の Observable（3×M）
    pos_obs = @lift begin
        i1 = max(1, $frame_0 - trail_len)
        @views pos_xyz[1:3, i1:$frame_0]
    end

    # 色（RGBA）の Observable（M要素）
    col_obs = @lift begin
        i1 = max(1, $frame_0 - trail_len)
        n  = $frame_0 - i1 + 1

        alphas = n <= 1 ? Float32[1] :
            Float32.(min.(range(0, 2; length=n), 1))

        rgb = RGBf(Makie.to_color(color))
        RGBAf.(rgb.r, rgb.g, rgb.b, alphas)
    end

    tail = lines!(ax, pos_obs; color=col_obs, linewidth, transparency=true)

    head = scatter!(ax,
        @lift(@views pos_xyz[1:3, $frame_0]'),  # 1×3 にして scatter に渡す
        markersize=markersize,
        color=Makie.to_color(color)             # ← ここがポイント（:colors じゃない）
    )

    return (tail=tail, head=head)
end

traj_fade! (generic function with 1 method)

In [6]:

show_propagated_flag = 0

if show_propagated_flag == 1
    N = size(earth_states,2)
    step_size_sec = 24*60*60

    results_dict = Dict{Int, NamedTuple{(:sol, :sol_em_fixed_e, :sol_lop_e), Tuple{Any, Any, Any}}}()

    for sol_from in 1:sampling_interval:(N-1)
        sol = solve_satellite(sc1_states, earth_states, moon_states, mars_states, jupiter_states;
            sol_from=sol_from, step_size_sec=step_size_sec, reltol=1e-10, abstol=1e-10)

        sol_N = size(sol,2)

        rel_moon_earth = moon_states[1:6, sol_from:sol_from+sol_N-1] .- earth_states[1:6, sol_from:sol_from+sol_N-1]
        rel_sc_earth   = sol[1:6, :]                  .- earth_states[1:6, sol_from:sol_from+sol_N-1]

        results_dict[sol_from] = (
            sol            = sol,
            sol_em_fixed_e = make_line_fixed_frame(rel_moon_earth, rel_sc_earth),
            sol_lop_e      = rotate_to_plane_fixed_t0(rel_moon_earth, rel_sc_earth)
        )
    end
end


In [14]:
using ProgressMeter

#描画領域の設定
set_theme!(theme_black()) # ダークテーマを設定
update_theme!(fontsize= 22)

title_ecliptic_s = "Fig.X Sun centered, Ecliptic of J2000"
title_ecliptic_e = "Fig.X Earth centered, Ecliptic of J2000"
title_ecliptic_j = "Fig.X Jupiter centered, Ecliptic of J2000"
title_ecliptic_europa = "Fig.X Europa centered, Ecliptic of J2000"

title_eme_e      = "Fig.X Earth centered, EME2000"
title_lop_e      = "Fig.1 Inertial frame"
title_se_fixed_e = "Fig.X Earth centered, Sun-Earth line fixed rotational frame"
title_em_fixed_e = "Fig.2 Earth-Moon line fixed rotational frame"
title_em_fixed_m = "Fig.x Moon centered, Earth-Moon line fixed rotational frame"

fig1 = Figure(size = (1400, 800))
fig2 = Figure(size = (1400, 800))

ax_lop_e      = Axis3(fig1[1, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_lop_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_em_fixed_e = Axis3(fig1[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_em_fixed_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)

ax_em_fixed_m = Axis3(fig2[2, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_em_fixed_m, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_s = Axis3(fig2[1, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_s, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_e = Axis3(fig2[1, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_j = Axis3(fig2[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_j, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_ecliptic_europa = Axis3(fig2[1, 2], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_ecliptic_europa, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_eme_e      = Axis3(fig2[1, 3], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_eme_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)
ax_se_fixed_e = Axis3(fig2[2, 1], aspect =:data, xlabel = "x", ylabel = "y", viewmode = :fit, clip=false, title = title_se_fixed_e, titlegap = -50, xspinesvisible=false,yspinesvisible=false,zspinesvisible=false, xticksvisible=false, yticksvisible=false, zticksvisible=false)

hidedecorations!(ax_lop_e)
hidedecorations!(ax_em_fixed_e)
hidedecorations!(ax_ecliptic_s)
hidedecorations!(ax_ecliptic_j)
hidedecorations!(ax_ecliptic_europa)

ax_lop_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_lop_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

ax_em_fixed_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_em_fixed_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

ax_ecliptic_s.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_s.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_e.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_e.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_j.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_j.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π
ax_ecliptic_europa.azimuth[]   =  -0.5 * pi #月から地球を見る方向が0π、数字大きくなると半時計周り方向へ
ax_ecliptic_europa.elevation[] =  0.5 * pi #水平から見るときは0π、頂点から見るときは0.5π

scale_ecliptic_s = 6e8
scale_ecliptic_e = 3e4
scale_ecliptic_j = 3e6
scale_ecliptic_europa = 2e4
scale_lop_e = 3e5
scale_lop_e = 9e5
scale_em_fixed_e = 2e5
scale_em_fixed_m = 7e4

limits!(ax_ecliptic_s, (-scale_ecliptic_s, scale_ecliptic_s), (-scale_ecliptic_s, scale_ecliptic_s), (-scale_ecliptic_s, scale_ecliptic_s))
limits!(ax_ecliptic_e, (-scale_ecliptic_e, scale_ecliptic_e), (-scale_ecliptic_e, scale_ecliptic_e), (-scale_ecliptic_e, scale_ecliptic_e))
limits!(ax_ecliptic_j, (-scale_ecliptic_j, scale_ecliptic_j), (-scale_ecliptic_j, scale_ecliptic_j), (-scale_ecliptic_j, scale_ecliptic_j))
limits!(ax_ecliptic_europa, (-scale_ecliptic_europa, scale_ecliptic_europa), (-scale_ecliptic_europa, scale_ecliptic_europa), (-scale_ecliptic_europa, scale_ecliptic_europa))
limits!(ax_lop_e, (-scale_lop_e, scale_lop_e), (-scale_lop_e, scale_lop_e), (-scale_lop_e, scale_lop_e))
limits!(ax_em_fixed_e, (-scale_em_fixed_e+2e5, scale_em_fixed_e+2e5), (-scale_em_fixed_e, scale_em_fixed_e), (-scale_em_fixed_e, scale_em_fixed_e))
limits!(ax_em_fixed_m, (-scale_em_fixed_m, scale_em_fixed_m), (-scale_em_fixed_m, scale_em_fixed_m), (-scale_em_fixed_m, scale_em_fixed_m))

######### 描画 ##################################################

sun_sphere   = Sphere(Point3f(0.0, 0.0, 0.0), 696340)
earth_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 6378)
moon_sphere  = Sphere(Point3f(0.0, 0.0, 0.0), 1737)
jupiter_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 69911)
europa_sphere = Sphere(Point3f(0.0, 0.0, 0.0), 1600)

#mesh!(ax_ecliptic_s, sun_sphere, color=:white)
mesh!(ax_ecliptic_e, earth_sphere, color=:cyan)
mesh!(ax_eme_e, earth_sphere, color=:cyan)
mesh!(ax_lop_e, earth_sphere, color=:cyan)
mesh!(ax_se_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_e, earth_sphere, color=:cyan)
mesh!(ax_em_fixed_m, moon_sphere, color=:yellow)
mesh!(ax_ecliptic_j, jupiter_sphere, color=:orange)
mesh!(ax_ecliptic_europa, europa_sphere, color=:cyan)
scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0, markersize=10, color=:white)

animation_flag = 0
sampling_interval1  = 60
#sampling_interval2 = 60
N = size(sc1_states, 2)
frames = 1:sampling_interval1:(N-1)

# 切り替えフレーム（ここから細かくする）
#switch_frame = Int(floor(0.6 * N))   # 例：後半20%を細かく
#frames1 = 1:sampling_interval1:(switch_frame-1)
#frames2 = switch_frame:sampling_interval2:(N-1)
#frames = vcat(frames1, frames2)

if animation_flag == 0

    lines!(ax_ecliptic_s, earth_states[1:3, :], linewidth=1, color=:cyan)
    lines!(ax_ecliptic_s, mars_states[1:3, :], linewidth=1, color=:red)
    lines!(ax_ecliptic_s, jupiter_states[1:3, :], linewidth=1, color=:orange)
    lines!(ax_ecliptic_s, sc1_states[1:3, :],  linewidth=1, color=:lime)

    lines!(ax_ecliptic_e, moon_states[1:3, :]- earth_states[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :],  linewidth=1, color=:lime)
    scatter!(ax_ecliptic_e, sc1_states[1:3, :]'- earth_states[1:3, :]',  markersize=10, color=:lime)
    
    lines!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :],  linewidth=1, color=:white)

    lines!(ax_eme_e, moon_states_eme[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_eme_e, sc1_states_eme[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_lop_e, moon_lop_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_lop_e, sc1_lop_e[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_lop_e, results_dict[651].sol_lop_e[1:3, :],  linewidth=1, color=:red)

    lines!(ax_se_fixed_e, moon_se_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_se_fixed_e, sc1_se_fixed_e[1:3, :],  linewidth=1, color=:lime)
    
    lines!(ax_em_fixed_e, moon_em_fixed_e[1:3, :], linewidth=1, color=:yellow)
    lines!(ax_em_fixed_e, sc1_em_fixed_e[1:3, :],  linewidth=1, color=:lime)
    #lines!(ax_em_fixed_e, results_dict[651].sol_em_fixed_e[1:3, :],  linewidth=1, color=:red)

    lines!(ax_em_fixed_m, sc1_em_fixed_m[1:3, :],  linewidth=1, color=:lime)

else
    frame_0   = Observable(0)
    frame_0[] = 1
    trail_len_long = 12000
    trail_len_short = 1200

    if show_propagated_flag == 1
        lines!(ax_ecliptic_s, @lift(results_dict[$frame_0].sol[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_lop_e, @lift(results_dict[$frame_0].sol_lop_e[1:3, :]),  linewidth=1, color=:red)
        lines!(ax_em_fixed_e, @lift(results_dict[$frame_0].sol_em_fixed_e[1:3, :]),  linewidth=1, color=:red)
    end

    text!(ax_ecliptic_s, 0.95, 0.85, text = @lift(Dates.format(t_tdb[$frame_0], "yyyy-mm-dd")), offset=(0, 0), align=(:right, :top), space=:relative,fontsize = 22)

    traj_fade!(ax_ecliptic_s, earth_states[1:3, :], frame_0; trail_len=trail_len_long, color=:cyan)
    traj_fade!(ax_ecliptic_s, mars_states[1:3, :], frame_0; trail_len=trail_len_long, color=:pink)
    traj_fade!(ax_ecliptic_s, jupiter_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)
    traj_fade!(ax_ecliptic_s, sc1_states[1:3, :], frame_0; trail_len=trail_len_long, color=:lime)
    #traj_fade!(ax_ecliptic_s, ryugu_states[1:3, :], frame_0; trail_len=trail_len_long, color=:orange)

    traj_fade!(ax_ecliptic_e, sc1_states[1:3, :]- earth_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    traj_fade!(ax_ecliptic_j, sc1_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:lime)
    #traj_fade!(ax_ecliptic_j, europa_states[1:3, :]- jupiter_states[1:3, :], frame_0; trail_len=trail_len_short, color=:white)

    traj_fade!(ax_lop_e, moon_lop_e, frame_0; trail_len=trail_len_short, color=:yellow)    
    traj_fade!(ax_lop_e, sc1_lop_e, frame_0; trail_len=trail_len_short, color=:lime)

    #moon_sphere_ine = lift((x, y, z) -> Sphere(Point3f(x, y, z), moon_radius*5),
    #@lift(Float32(moon_int[1, $frame_0])),
    #@lift(Float32(moon_int[2, $frame_0])),
    #@lift(Float32(moon_int[3, $frame_0])))
    #mesh!(ax_ine, moon_sphere_ine, color=:yellow)

    p = Progress(length(frames); desc = "Rendering movie")

    # アニメ出力 (interpolationed) 
    record(fig1, "../output/$(spacecraft)_5.mp4", frames;framerate = 30) do frame
    #framerateは1秒間のフレーム数(60fpsが標準的な上限) #地球火星の軌道を描くための設定(frame_planetを更新) 

        frame_0[] = frame
        next!(p)   # ← これを1行追加

    end

end

fig1


In [8]:
#= 
####地球月固定回転座標(地球中心)##########################################################################

# z軸を設定
z_axis = repeat([0.0, 0.0, 1.0], 1, size(moon_states, 2))
# 月公転面法線単位ベクトルを計算
normal_vec  = calculate_normalized_cross_product(moon_states[1:3, :], moon_states[4:6, :])
# 月公転面法線単位ベクトルと(0,0,1)の外積を計算
rotation_axis = calculate_normalized_cross_product(normal_vec, z_axis)
# 月公転面法線単位ベクトルと(0,0,1)の角度alpha_eを計算
alpha = calculate_angle(normal_vec,z_axis)
# 地球月固定回転座標(地球中心)中間生成物 (軌道面を常にxy平面に平行にする回転)
moon_rot_tochu = rotate_coordinates(moon_states[1:3, :], cos.(alpha), sin.(alpha), rotation_axis)
# 地球月固定回転座標(地球中心)
earth_moon_dist = sqrt.(moon_states[1,:].^2+moon_states[2,:].^2+moon_states[3,:].^2)
cos_θ_val = moon_rot_tochu[1,:]./earth_moon_dist[:]
sin_θ_val = -moon_rot_tochu[2,:]./earth_moon_dist[:]
moon_rot = rotate_coordinates(moon_rot_tochu, cos_θ_val, sin_θ_val,z_axis)


#####地球中心座標 (月の軌道傾斜角を「ほぼ0度」で表示したい場合) #############################################

#非回転座標で月の傾斜角が気になるので実装。ただし、t=0におけるデータの値を用いて回転させている。
#理由は軌道面は少なからず変動しており、時間ごとにすべて軌道傾斜角を0にする変換をしてしまうと、軌道傾斜角の変化が見えない。
#つまり実際に見えるものとは違ってしまう。そのためt=0のデータで回転させている。一方、地球月固定回転座標では
#中間生成物として時間ごとに軌道傾斜角を0にしている。理由は、earth-moon line fixedを計算するために必要な処理であるため。

N = size(moon_states, 2)
alpha_fixed = fill(alpha[1],N) #t=0における回転角度
rotation_axis_fixed = repeat(rotation_axis[:, 1], 1, N)
moon_lop = rotate_coordinates(moon_states[1:3, :], cos.(alpha_fixed), sin.(alpha_fixed), rotation_axis_fixed) #lunar orbital plane
=#



In [9]:
#=
tt1 = TTEpoch("2018-01-01T12:00:00")
tt2 = TTEpoch(2018, 1, 1, 12, 0, 0.0)
tai = TAIEpoch(tt)
tdb1 = TDBEpoch(tai)
tdb2 = TDBEpoch(tt)
tt3 = TTEpoch(tdb2)

println(tt1)
println(tt2)
println(tai)
println(tdb1)
println(tdb2)
println(tt3)
=#

In [10]:
#=
# ============================================================
# earthの型と構造を確認
# ============================================================

# 型を確認
println("earthの型:")
println(typeof(earth))
println()

# NamedTupleの場合、フィールド名を確認
if typeof(earth) <: NamedTuple
    println("フィールド名:")
    println(fieldnames(typeof(earth)))
    println()
    
    # 各フィールドの型と要素数を確認
    println("各フィールドの詳細:")
    for name in fieldnames(typeof(earth))
        value = getfield(earth, name)
        println("  $name: $(typeof(value)), 要素数: $(length(value))")
    end
    println()
    
    # 最初の数個のデータを表示（サンプル）
    println("データのサンプル（最初の3要素）:")
    println("  t (時刻): ", earth.t[1:min(3, length(earth.t))])
    println("  cal (カレンダー): ", earth.cal[1:min(3, length(earth.cal))])
    println("  x (X座標): ", earth.x[1:min(3, length(earth.x))])
    println("  y (Y座標): ", earth.y[1:min(3, length(earth.y))])
    println("  z (Z座標): ", earth.z[1:min(3, length(earth.z))])
    println("  ẋ (X速度): ", earth.ẋ[1:min(3, length(earth.ẋ))])
    println("  ẏ (Y速度): ", earth.ẏ[1:min(3, length(earth.ẏ))])
    println("  ż (Z速度): ", earth.ż[1:min(3, length(earth.ż))])
end
=#

In [11]:
#=
using EphemerisSources
using AstroTime, Dates

# ============================================================
# 複数の天体・人工衛星のエフェメリス（位置・速度データ）を取得
# ============================================================

# 日時範囲の設定（既存の変数を使用、または新しく定義）
# start_date, stop_date を start, stop として使用

# ============================================================
# AstroTime.TTEpoch について
# ============================================================
# AstroTimeは天文計算用の時間表現を扱うJuliaパッケージです。
# 
# TTEpoch（Terrestrial Time Epoch）の処理内容：
# 1. 時刻の変換: 通常のDateTime（UTCやローカル時間）を
#    Terrestrial Time（TT、地球時）に変換します
# 2. 時系の統一: 天文計算では様々な時系（UTC, TAI, TT, TDB等）が使われますが、
#    TTEpochはTT（地球時）を基準とした統一的な時刻表現を提供します
# 3. エポック表現: 天文計算でよく使われる「エポック」（特定の基準時刻からの経過時間）
#    の形式で時刻を表現します
#
# なぜTTを使うのか：
# - TT（Terrestrial Time）は地球表面での時間の流れを表す時系で、
#   天文計算の標準的な時系として広く使われています
# - UTC（協定世界時）はうるう秒の挿入により不連続ですが、
#   TTは連続的な時系のため、精密な軌道計算に適しています
# - EphemerisSourcesパッケージのephemeris関数は、TT形式の時刻を期待します
#
# 処理の流れ：
#   now() → DateTime（現在のUTC時刻）
#   ↓
#   AstroTime.TTEpoch() → TTEpoch（TT形式のエポック）
#   ↓
#   ephemeris関数で使用可能な形式
# ============================================================


# ============================================================
# 複数の天体・人工衛星のエフェメリス（位置・速度データ）を取得
# ============================================================

# 日時範囲の設定（既存の変数を使用、または新しく定義）
# start_date, stop_date を start, stop として使用

# ============================================================
# AstroTime.TTEpoch について
# ============================================================
# AstroTimeは天文計算用の時間表現を扱うJuliaパッケージです。
# 
# TTEpoch（Terrestrial Time Epoch）の処理内容：
# 1. 時刻の変換: 通常のDateTime（UTCやローカル時間）を
#    Terrestrial Time（TT、地球時）に変換します
# 2. 時系の統一: 天文計算では様々な時系（UTC, TAI, TT, TDB等）が使われますが、
#    TTEpochはTT（地球時）を基準とした統一的な時刻表現を提供します
# 3. エポック表現: 天文計算でよく使われる「エポック」（特定の基準時刻からの経過時間）
#    の形式で時刻を表現します
#
# なぜTTを使うのか：
# - TT（Terrestrial Time）は地球表面での時間の流れを表す時系で、
#   天文計算の標準的な時系として広く使われています
# - UTC（協定世界時）はうるう秒の挿入により不連続ですが、
#   TTは連続的な時系のため、精密な軌道計算に適しています
# - EphemerisSourcesパッケージのephemeris関数は、TT形式の時刻を期待します
#
# 処理の流れ：
#   now() → DateTime（現在のUTC時刻）
#   ↓
#   AstroTime.TTEpoch() → TTEpoch（TT形式のエポック）
#   ↓
#   ephemeris関数で使用可能な形式
# ============================================================
# ============================================================
# AstroTime. の意味と処理内容
# ============================================================
# `AstroTime.` は名前空間（namespace）の指定です。
#
# 処理内容：
# 1. パッケージの明示: `AstroTime.TTEpoch` と書くことで、
#    AstroTimeパッケージのTTEpoch型/コンストラクタを明示的に指定します
# 2. 名前の解決: `using AstroTime` をしているので、通常は `TTEpoch` だけで使えますが、
#    `AstroTime.TTEpoch` と書くことで、どのパッケージのTTEpochを使っているかを明確にします
# 3. 曖昧さの回避: 複数のパッケージに同名の型がある場合、明示的に指定することで
#    エラーを防ぎます
#
# 使用例の比較：
# - `TTEpoch(...)` : `using AstroTime` により、AstroTimeパッケージのTTEpochを使用
# - `AstroTime.TTEpoch(...)` : 明示的にAstroTimeパッケージのTTEpochを指定
# どちらも同じ結果になりますが、後者の方がコードの可読性が高い場合があります
# ============================================================

println("2026-01-24T13:50:31")
println(TTEpoch("2026-01-24T13:50:31"))           # 名前空間を省略した書き方
println(AstroTime.TTEpoch("2026-01-24T13:50:31")) # 名前空間を明示した書き方

start1 = "2026-01-24T13:50:31"                    # String型
start2 = TTEpoch("2026-01-24T13:50:31")           # TTEpoch型（名前空間省略）
start3 = AstroTime.TTEpoch("2026-01-24T13:50:31") # TTEpoch型（名前空間明示）

println(typeof(start1))
println(typeof(start2))
println(typeof(start3))

start = TTEpoch("2026-01-24T13:50:31")
stop = start + 1days

println(stop)

# 時間刻みの設定
# "1 day" は1日間隔でデータを取得することを意味します
# 他の例: "1 hour" (1時間), "6 hours" (6時間), "1 min" (1分) なども指定可能
step = "1 min"

# 基準座標系（wrt: with respect to）の設定
# "SEMB-L2" は Sun-Earth-Moon Barycenter (太陽-地球-月の重心) を基準とした
# ラグランジュ点L2を中心とする座標系を意味します
# この座標系は、太陽-地球-月系の重心を基準として、L2点を原点とする座標系です
wrt = "SEMB-L2"

# 単位の設定
# "KM-S" は距離をキロメートル(km)、速度をキロメートル毎秒(km/s)で取得することを意味します
# 他の例: "AU-D" (距離:天文単位, 速度:天文単位/日) なども指定可能
units = "KM-S"

# 地球のエフェメリスを取得
# ephemeris関数は、指定された天体の位置・速度データを時系列で取得します
# 第1引数: 天体名（"earth"は地球のHorizons ID）
# 第2-4引数: 開始日時、終了日時、時間刻み
# キーワード引数:
#   - wrt: 基準座標系（上記で設定）
#   - units: 単位系（上記で設定）
# 戻り値: 時刻、位置ベクトル、速度ベクトルを含むデータ構造
earth = ephemeris("earth", start, stop, step; wrt=wrt, units=units)
tess = ephemeris("TESS", start, stop, step; wrt=wrt, units=units)


# 時間刻みの設定
# "1 day" は1日間隔でデータを取得することを意味します
# 他の例: "1 hour" (1時間), "6 hours" (6時間), "1 min" (1分) なども指定可能
step = "1 min"

# 基準座標系（wrt: with respect to）の設定
# "SEMB-L2" は Sun-Earth-Moon Barycenter (太陽-地球-月の重心) を基準とした
# ラグランジュ点L2を中心とする座標系を意味します
# この座標系は、太陽-地球-月系の重心を基準として、L2点を原点とする座標系です
wrt = "SEMB-L2"

# 単位の設定
# "KM-S" は距離をキロメートル(km)、速度をキロメートル毎秒(km/s)で取得することを意味します
# 他の例: "AU-D" (距離:天文単位, 速度:天文単位/日) なども指定可能
units = "KM-S"

# 地球のエフェメリスを取得
# ephemeris関数は、指定された天体の位置・速度データを時系列で取得します
# 第1引数: 天体名（"earth"は地球のHorizons ID）
# 第2-4引数: 開始日時、終了日時、時間刻み
# キーワード引数:
#   - wrt: 基準座標系（上記で設定）
#   - units: 単位系（上記で設定）
# 戻り値: 時刻、位置ベクトル、速度ベクトルを含むデータ構造
earth = ephemeris("earth", start, stop, step; wrt=wrt, units=units)
tess = ephemeris("TESS", start, stop, step; wrt=wrt, units=units)
=#

